In [ ]:
!pip install x-transformers pythainlp gensim --quiet

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import Counter

from x_transformers import TransformerWrapper, Decoder
from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize
from pythainlp.word_vector import WordVector

In [ ]:
RANDOM_STATE = 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 100

HIDDEN_DIM = 300
DEPTH = 4
HEADS = 4
MAX_LEN = 512
MAX_VOCAB = 20000

PAD_ID = 0
UNK_ID = 1

labels = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

NUM_CLASSES = len(labels)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

In [ ]:
df = pd.read_csv("/kaggle/input/sied-thai/SIED-Thai.csv")

df = df[['Tweets'] + labels].dropna()

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df['text'] = df['Tweets'].apply(clean_text)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class EmotionDataset(Dataset):

    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df[labels].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

train_loader = DataLoader(EmotionDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(EmotionDataset(val_df), batch_size=BATCH_SIZE)
test_loader  = DataLoader(EmotionDataset(test_df), batch_size=BATCH_SIZE)

In [ ]:
class Model(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

        self.decoder = Decoder(
            dim=DECODER_DIM,
            depth=DECODER_DEPTH,
            heads=DECODER_HEADS
        )

        self.fc = nn.Linear(DECODER_DIM, NUM_CLASSES)

    def forward(self, input_ids, attention_mask):

        x = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        x = self.decoder(
            x,
            mask=attention_mask.bool()
        )

        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(1) / mask.sum(1)

        return self.fc(x)

model = Model().to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()


In [ ]:
@torch.no_grad()
def evaluate(loader, threshold=0.5):

    model.eval()

    preds = []
    targets = []

    for input_ids, attention_mask, y in loader:

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)

        probs = torch.sigmoid(logits)
        pred = (probs > threshold).int().cpu().numpy()

        preds.append(pred)
        targets.append(y.numpy())

    y_pred = np.vstack(preds)
    y_true = np.vstack(targets)

    hamming_acc = (y_pred == y_true).mean()
    exact_acc = (y_pred == y_true).all(axis=1).mean()

    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")

    precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)

    recall_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)

    return (
        hamming_acc, exact_acc,
        f1_micro, f1_macro,
        precision_micro, precision_macro,
        recall_micro, recall_macro
    )


In [ ]:
@torch.no_grad()
def evaluate_per_class(loader, threshold=0.5):

    model.eval()

    preds = []
    targets = []

    for input_ids, attention_mask, y in loader:

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)

        probs = torch.sigmoid(logits)
        pred = (probs > threshold).int().cpu().numpy()

        preds.append(pred)
        targets.append(y.numpy())

    y_pred = np.vstack(preds)
    y_true = np.vstack(targets)

    print("\n=== Per-Class Metrics ===")
    print(f"{'Label':<20} {'Precision':>10} {'Recall':>10} {'F1-score':>10}")

    for i, label in enumerate(labels):
        p = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
        r = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)
        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)

        print(f"{label:<20} {p*100:10.2f} {r*100:10.2f} {f1*100:10.2f}")

In [ ]:
for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for input_ids, attention_mask, y in train_loader:

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    h_acc, e_acc, f1_micro, f1_macro, _, _, _, _ = evaluate(val_loader)

    print(
        f"Epoch {epoch+1} | "
        f"loss={total_loss:.4f} | "
        f"h_acc={h_acc:.4f} | "
        f"exact_acc={e_acc:.4f} | "
        f"f1_micro={f1_micro:.4f} | "
        f"f1_macro={f1_macro:.4f}"
    )


In [ ]:
h_acc, e_acc, f1_micro, f1_macro, p_micro, p_macro, r_micro, r_macro = evaluate(test_loader)

print("\nFINAL TEST")
print(f"Hamming Acc   : {h_acc:.4f}")
print(f"Exact Acc     : {e_acc:.4f}")
print(f"F1 Micro      : {f1_micro:.4f}")
print(f"F1 Macro      : {f1_macro:.4f}")
print(f"Precision Mic : {p_micro:.4f}")
print(f"Precision Mac : {p_macro:.4f}")
print(f"Recall Mic    : {r_micro:.4f}")
print(f"Recall Mac    : {r_macro:.4f}")

evaluate_per_class(test_loader)